# Forecast walkthrough — CVE → will-it-be-exploited?

**Task:** at/near publication, estimate the probability a CVE will be exploited in the wild.
**Scoring:** Brier (stored as `1 − Brier`), plus run-level ROC AUC. **Leak-resistance:** HIGH —
the answer doesn't exist at submission time; it's resolved later by CISA KEV.

In [ ]:
import os, sys, json

def _find_root(start):
    d = os.path.abspath(start)
    while d != os.path.dirname(d):
        if os.path.isdir(os.path.join(d, "src", "glokta")):
            return d
        d = os.path.dirname(d)
    raise RuntimeError("could not locate repo root (a dir containing src/glokta)")

ROOT = _find_root(os.getcwd())
sys.path.insert(0, os.path.join(ROOT, "src"))

# Best-effort load of the repo .env so live HF calls have HF_TOKEN; no dotenv dependency.
_envp = os.path.join(ROOT, ".env")
if os.path.exists(_envp):
    for _line in open(_envp):
        _s = _line.strip()
        if _s and not _s.startswith("#") and "=" in _s:
            _k, _v = _s.split("=", 1)
            os.environ.setdefault(_k.strip(), _v.strip())
os.environ.setdefault("TESTING", "1")  # relax settings validators if .env is absent

MODEL = "huggingface/meta-llama/Llama-3.1-8B-Instruct"
LIVE = bool(os.environ.get("HF_TOKEN"))
print("repo root :", ROOT)
print("model     :", MODEL)
print("LIVE calls:", LIVE, "(set HF_TOKEN to enable real inference)")

def run_model(prompt, canned, max_tokens=256):
    """Call the model live if HF_TOKEN is set, else return a canned example response."""
    if LIVE:
        from glokta.infrastructure.cti.inference import complete
        try:
            return complete(MODEL, prompt, max_tokens=max_tokens, timeout=60.0, max_retries=1)
        except Exception as exc:
            print("[live call failed -> canned]", type(exc).__name__, str(exc)[:80])
            return canned
    print("[offline -> canned response]")
    return canned

## 1. Dataflow — seed at publication, resolve later via KEV
An item is seeded with a provisional `exploited=False` label; when the CVE appears in KEV it flips to `True` and the temporal anchor advances to the KEV date.

In [ ]:
# A CVE JSON 5.0 record (cvelistV5 shape). In production these come from the delta feed
# (fetch_recent_cve_records); here we use a representative in-memory example.
CVE_RECORD = {
    "cveMetadata": {"cveId": "CVE-2024-12345", "datePublished": "2024-05-01T10:00:00.000Z",
                    "dateUpdated": "2024-05-20T10:00:00.000Z", "state": "PUBLISHED"},
    "containers": {
        "cna": {
            "descriptions": [{"lang": "en",
                "value": "A SQL injection vulnerability in Acme Portal allows a remote "
                         "unauthenticated attacker to execute arbitrary SQL via the search parameter."}],
            "problemTypes": [{"descriptions": [{"lang": "en", "cweId": "CWE-89",
                              "description": "SQL Injection"}]}],
            "metrics": [{"cvssV3_1": {"vectorString": "CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H",
                                       "baseScore": 9.8}}],
        },
        # CISA-ADP (Vulnrichment) container — here it agrees with the CNA on the CWE.
        "adp": [{"providerMetadata": {"shortName": "CISA-ADP", "dateUpdated": "2024-05-20T10:00:00.000Z"},
                 "problemTypes": [{"descriptions": [{"lang": "en", "cweId": "CWE-89"}]}]}],
    },
}

In [ ]:
from datetime import date
# At publication: the input is the CVE description; the label is unknown -> provisional False.
desc = CVE_RECORD["containers"]["cna"]["descriptions"][0]["value"]
label_unresolved = {"exploited": False}
# Later: CISA KEV lists the CVE -> resolve_forecast_labels flips it and advances the anchor.
label_resolved = {"exploited": True}   # anchor would move to max(publication, KEV dateAdded)
print("input (CVE desc):", desc[:90], "...")
print("label at submission :", label_unresolved, "(leak-proof: answer not yet known)")
print("label after KEV     :", label_resolved)

## 2. Tasking — prompt + model call (a probability)

In [ ]:
from glokta.infrastructure.cti.prompts import build_prompt, parse_response

prompt = build_prompt("forecast", desc)
print(prompt)
print("-" * 70)
response = run_model(prompt, canned="Given network-exploitable RCE, I estimate 0.85")
print("model response :", repr(response))
print("parsed prob    :", parse_response("forecast", response))

## 3. Scoring — Brier against the resolved outcome
`evaluate_item` stores `1 − Brier`; `correct` is whether the >0.5 call matched the outcome.

In [ ]:
from glokta.infrastructure.cti.evaluator import evaluate_item

for outcome_name, label in {"exploited (KEV)": label_resolved, "not exploited": label_unresolved}.items():
    for pname, resp in {"confident 0.85": "0.85", "uncertain 0.5": "0.5", "low 0.1": "0.1"}.items():
        s = evaluate_item("forecast", label, resp)
        print(f"{outcome_name:16} {pname:14} score(1-brier)={s.score:.3f} correct={s.correct}")

## 4. Run-level AUC
Per item we store Brier; across a run we also compute ROC AUC (`auc_for_run`). AUC needs both
classes present, so it's `None` for a single-class slice.

In [ ]:
from types import SimpleNamespace
from glokta.application.cti.scoring_aggregate import auc_for_run

# Simulate a few scored forecast results (score_breakdown carries prob + outcome).
fake = [SimpleNamespace(score_breakdown={"prob": p, "outcome": o})
        for p, o in [(0.9, 1.0), (0.8, 1.0), (0.2, 0.0), (0.1, 0.0)]]
print("AUC (perfect separation):", auc_for_run(fake))
print("AUC (single class)      :", auc_for_run(fake[:2]))

**Takeaway:** Forecast is the structurally leak-proof task — a model evaluated at publication cannot have memorised an answer that KEV only assigns weeks later. Advancing the temporal anchor to the KEV date keeps post-cutoff outcomes from being mis-tagged as pre-cutoff.